# Learn-a-World v3: Phase 3 T1 Joint Retrain (Repair Protocol)
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sahiltolani30/learn-a-world/blob/main/notebooks/05_phase3_t1_joint_repair.ipynb)

**Campaign Context & Core Directive:**
- **Ground Truth**: Google Drive is the sole source of truth for all data, manifests, and checkpoints.
- **Root Cause Remediation**: Curiosity-only retraining caused policy drift on combat (`defeat_zombie`: 20% -> 0%). This notebook implements **Joint Retraining** on all four audited data families:
  1. `structured_random` (ONLY `_final` cumulative snapshot: ~150k transitions)
  2. `teacher_hybrid` (ONLY `_final` cumulative snapshot or largest available e.g. `0009`: ~100k-150k transitions)
  3. `planner_v2` (ONLY `_final` cumulative snapshot: ~30k transitions)
  4. `curiosity_v3` (ALL 76 disjoint exploration shards: ~160k transitions)
- **Total Joint Buffer**: ~440,000 – 490,000 transitions.
- **Starting Checkpoint**: Strictly `checkpoints/jepa_crafter_v3_phase2.pt` (pre-Phase-3 baseline).
- **Target Bar**: Crafter Score > **2.313%** AND `defeat_zombie` > **0.0%** (Real target: **2.74%** v2 champion).

---

### Step 1: Mount Google Drive
Mounts Google Drive to access all ground-truth datasets, checkpoints, and logs.

In [1]:
from google.colab import drive
from pathlib import Path

# Mount Google Drive
drive.mount('/content/drive')

DRIVE_DIR = '/content/drive/MyDrive/learn-a-world'
drive_p = Path(DRIVE_DIR)
(drive_p / 'checkpoints').mkdir(parents=True, exist_ok=True)
(drive_p / 'results').mkdir(parents=True, exist_ok=True)
(drive_p / 'data' / 'combined_shards_run3').mkdir(parents=True, exist_ok=True)

print('Google Drive connected at:', DRIVE_DIR)

Mounted at /content/drive
Google Drive connected at: /content/drive/MyDrive/learn-a-world


### Step 2: Clone or Pull Repository on Branch `main` & Install Requirements

In [2]:
%%bash
cd /content
if [ ! -d "learn-a-world" ]; then
  git clone https://github.com/sahiltolani30/learn-a-world.git
fi
cd learn-a-world
git fetch origin
git checkout main
git pull origin main
echo "--> Latest Git Commit:"
git log -1 --oneline

pip install -q -r requirements.txt
pip install -q crafter gymnasium pytest
nvidia-smi

Your branch is up to date with 'origin/main'.
Already up to date.
--> Latest Git Commit:
4e90515 fix(colab): zero-RAM Step 6 verify_joint_shards and single buffer load in Step 7 to prevent OOM crash
Fri Oct 23 19:06:28 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   41C    P8             10W

Cloning into 'learn-a-world'...
Already on 'main'
From https://github.com/sahiltolani30/learn-a-world
 * branch            main       -> FETCH_HEAD


### Step 3: Run Pre-Flight Tests
Validates that all architecture and collector unit tests pass.

In [ ]:
!cd /content/learn-a-world && python3 -m pytest tests/test_phase3_curiosity.py -v

### Step 4: Step 0 & 1 Audits on Drive (Accounting & Manifest)
Inspects Drive logs directly:
1. Accounts for tonight's run in `step3_t1_train_results.json`.
2. Recounts curiosity shards (0000-0075) and exact `defeat_zombie` events from Drive manifest.

In [3]:
import sys
sys.path.insert(0, '/content/learn-a-world')
from training.run_v3_colab_run3 import audit_tonight_run, audit_curiosity_manifest

# 1. Account for tonight's run
train_audit = audit_tonight_run(DRIVE_DIR)

# 2. Audit curiosity shards & recount defeat_zombie
curiosity_audit = audit_curiosity_manifest(DRIVE_DIR)

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)



STEP 0 AUDIT: TONIGHT'S RETRAIN ACCOUNTING


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


[Audit] Found: /content/drive/MyDrive/learn-a-world/results/step3_t1_train_results.json
  Training Steps:    15000
  Base Checkpoint:   N/A
  Output Checkpoint: N/A
  Shards Directory:  N/A
  Recorded MSE:      0.8321744700272878

  Buffer Composition: No explicit composition dict embedded.
  Buffer size from tonight's log: ~167,976 transitions (80 shards).

  [VERDICT] This run was CURIOSITY-ONLY.
            Replayed ≤ 2,000 base teacher transitions alongside ~160k curiosity transitions.
            This caused policy forgetting on combat (defeat_zombie: 20% -> 0%).

STEP 1 & 2 AUDIT: DRIVE CURIOSITY SHARDS & MANIFEST
[Audit] Curiosity directory: /content/drive/MyDrive/learn-a-world/data/curiosity_shards
  Found 76 .npz shard files
  First shard: shard_curiosity_v3_t01_0000.npz
  Last shard:  shard_curiosity_v3_t01_0075.npz

  Sample Shard (shard_curiosity_v3_t01_0000.npz) NPZ Keys:
    Keys: ['obs', 'actions', 'rewards', 'dones', 'events', 'mask_before', 'episode_lengths', 'sources'

### Step 5: Step 1 Commit — Save Curiosity-Only Screening Record UNCHANGED
Commits `results/screen_v3_t1_results.json` from Drive to GitHub unchanged (2.0488% curiosity-only record).

In [ ]:
%%bash
cd /content/learn-a-world
if [ -f "/content/drive/MyDrive/learn-a-world/results/screen_v3_t1_results.json" ]; then
  cp /content/drive/MyDrive/learn-a-world/results/screen_v3_t1_results.json results/
  git config --global user.name "Sahil Tolani"
  git config --global user.email "sahiltolani30@gmail.com"
  git add results/screen_v3_t1_results.json
  git commit -m "chore: record Phase 3 T1 curiosity-only screening results (2.0488% score)" || true
  git push origin main || true
  echo "--> Committed and pushed results/screen_v3_t1_results.json"
else
  echo "Notice: screen_v3_t1_results.json not found on Drive"
fi

### Step 6: Step 2 & 3 Drive Base Shard Discovery & Joint Buffer Verification (Zero-RAM Audit)
Finds the 3 cumulative `_final` base shards directly on Drive and runs a zero-RAM shape audit.
- Structured Random: ONLY `_final` (~150,000 transitions)
- Teacher Hybrid: ONLY `_final` or largest available snapshot e.g. `0009` (~100k-150k transitions)
- Planner v2: ONLY `_final` (~30,000 transitions)
- Curiosity Shards: ALL 76 .npz files (~160,000 transitions)

**Hard Assertion**: Total transitions must be $\ge 400,000$ (with teacher snapshot) or $\ge 450,000$ (with final).

In [ ]:
from training.run_v3_colab_run3 import find_drive_base_shards, verify_joint_shards

# 1. Locate the 3 ground-truth base final shards on Drive
base_shards = find_drive_base_shards(DRIVE_DIR)

# 2. Zero-RAM verification (counts transitions without loading frames into RAM)
composition = verify_joint_shards(
    structured_random_final=base_shards['structured_random_final'],
    teacher_hybrid_final=base_shards['teacher_hybrid_final'],
    planner_v2_final=base_shards['planner_v2_final'],
    curiosity_dir=f'{DRIVE_DIR}/data/curiosity_shards',
)

print('Buffer verification passed! Total transitions:', composition['total_transitions'])


STEP 2: LOCATING BASE _FINAL SHARDS ON DRIVE
  ✅ structured_random_final   :  150,000 transitions  (data/sequences/shard_structured_random_final.npz)
  ✅ teacher_hybrid_final      :  150,000 transitions  (data/sequences/shard_teacher_hybrid_final.npz)
  ✅ planner_v2_final          :   30,000 transitions  (results/sequences/shard_planner_v2_final.npz)

JOINT SHARD VERIFICATION — T1 REPAIR (Zero-RAM Audit)
Cumulative Families : Verify _final snapshot per base family.
Disjoint Family     : Count ALL curiosity exploration shards.



### Step 7: Step 4 — Joint Retraining (15,000 Steps)
Trains JEPA v3 on the audited joint buffer:
- **Base Checkpoint**: Strictly `jepa_crafter_v3_phase2.pt` (never from T1 curiosity-only model).
- **Sampling**: `event_balanced` (generic inverse-frequency weighting without hardcoding).
- **Checkpoints**: Saves last and best checkpoint variants every 1,000 steps with full training state (optimizer, RNG, config).
- **Output**: `checkpoints/jepa_crafter_v3_t1_joint.pt` on Drive.

In [ ]:
from training.run_v3_colab_run3 import retrain_t1_joint

ckpt_path, buffer_composition = retrain_t1_joint(
    drive_dir=DRIVE_DIR,
    composition=composition,
    structured_random_final=base_shards['structured_random_final'],
    teacher_hybrid_final=base_shards['teacher_hybrid_final'],
    planner_v2_final=base_shards['planner_v2_final'],
    curiosity_dir=f'{DRIVE_DIR}/data/curiosity_shards',
    base_checkpoint=f'{DRIVE_DIR}/checkpoints/jepa_crafter_v3_phase2.pt',
    num_steps=15_000,
    batch_size=16,
    seq_len=16,
    lr=2e-4,
    device='cuda',
)

print('Joint retrain complete! Output checkpoint:', ckpt_path)

### Step 8: Step 5 — Screening Evaluation (50 Episodes, Seeds 2050–2099)
Evaluates the jointly retrained model on the untouched 50-episode test seeds.
Results are saved to `results/screen_v3_t1_joint_results.json` with embedded buffer composition.

In [ ]:
from training.run_v3_colab_run3 import screen_t1_joint

results = screen_t1_joint(
    checkpoint_path=ckpt_path,
    drive_dir=DRIVE_DIR,
    buffer_composition=buffer_composition,
    num_episodes=50,
    seed_start=2050,
    device='cuda',
)

### Step 9: Gate Check & Results Push to GitHub
Evaluates whether the checkpoint clears the T1 Gate:
- Crafter Score > 2.313%
- `defeat_zombie` > 0.0%

Commits only the raw JSON screening results to GitHub (keeps large binaries on Drive).

In [ ]:
from training.run_v3_colab_run3 import gate_check_t1_joint

# Evaluate gate
gate_passed = gate_check_t1_joint(results)

# Commit screening results to GitHub
!cd /content/learn-a-world && \
 git config --global user.name "Sahil Tolani" && \
 git config --global user.email "sahiltolani30@gmail.com" && \
 git add results/screen_v3_t1_joint_results.json && \
 git commit -m "results(phase3): record T1 joint retrain screening results" && \
 git push origin main